[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/onnx/tutorials/blob/main/02_Introduction_to_ONNX/03_ONNX_Ecosystem_Overview/ONNX_Ecosystem_Overview_Deep_Dive.ipynb)

# 1.3 ONNX Ecosystem Overview — Deep Dive

This notebook provides a rigorous, mathematically grounded tour of the **ONNX ecosystem**: the constellation of model collections, converters, runtimes, optimization tools, and hardware partnerships that surround the ONNX interchange format. We move beyond treating ONNX as a mere file format and study it as a **software ecosystem** with formal structure.

## Table of Contents
1. [Formal Taxonomy of the ONNX Ecosystem](#section-1)
2. [The Converter Pipeline Architecture](#section-2)
3. [Converters — Deep Dive](#section-3)
4. [ONNX Model Zoo — Curated Model Collections](#section-4)
5. [Runtime Landscape — ORT, TensorRT, OpenVINO](#section-5)
6. [Execution Provider Architecture](#section-6)
7. [Tools Ecosystem — Visualization, Validation, Optimization](#section-7)
8. [Ecosystem Growth Timeline](#section-8)
9. [The Full Ecosystem Interaction Graph](#section-9)
10. [Quantitative Analysis of Ecosystem Maturity](#section-10)
11. [Key Properties and Guarantees](#section-11)
12. [Summary and Connections](#section-12)

<a id='section-1'></a>
## Section 1: Formal Taxonomy of the ONNX Ecosystem

### Definition 1.1 (The ONNX Ecosystem)

The ONNX ecosystem $\mathcal{E}$ can be formally defined as a structured collection of six component sets:

$$\mathcal{E} = (\mathcal{P}, \mathcal{C}, \mathcal{R}, \mathcal{T}, \mathcal{Z}, \mathcal{H})$$

where:
- $\mathcal{P}$ = **Producers** — training frameworks that export to ONNX
- $\mathcal{C}$ = **Converters** — tools that translate framework-specific representations into ONNX
- $\mathcal{R}$ = **Runtimes** — inference engines that execute ONNX models
- $\mathcal{T}$ = **Tools** — utilities for inspection, validation, optimization, and debugging
- $\mathcal{Z}$ = **Model Zoo** — curated collections of pre-trained ONNX models
- $\mathcal{H}$ = **Hardware Partners** — silicon vendors with optimized backends

Each component set has well-defined interfaces and interaction patterns. The ecosystem operates as a directed pipeline: producers generate models through converters, which flow through optional tool transformations before reaching runtimes that dispatch to hardware-specific backends. This pipeline is analogous to a compiler toolchain: source languages → frontend → IR → optimizer → backend → machine code.

### The Component Interaction Relation

We define the **interaction relation** $\mathcal{I}$ on the ecosystem as a set of directed edges between component categories:

$$\mathcal{I} = \{(\mathcal{P}, \mathcal{C}), (\mathcal{C}, \text{ONNX}), (\text{ONNX}, \mathcal{T}), (\mathcal{T}, \text{ONNX}'), (\text{ONNX}', \mathcal{R}), (\mathcal{R}, \mathcal{H}), (\mathcal{Z}, \text{ONNX})\}$$

This yields the canonical data-flow chain:

$$\mathcal{P} \xrightarrow{\mathcal{C}} \text{ONNX} \xrightarrow{\mathcal{T}} \text{ONNX}_{\text{opt}} \xrightarrow{\mathcal{R}} \mathcal{H} \to \text{Output}$$

### Ecosystem Cardinality

As of 2024, the approximate sizes of each component set are:

| Component Set | Symbol | $|\cdot|$ | Key Members |
|:---|:---:|:---:|:---|
| Producers | $\mathcal{P}$ | $\approx 12$ | PyTorch, TensorFlow, JAX, scikit-learn, XGBoost, LightGBM, Keras, PaddlePaddle, MXNet, MATLAB, Spark ML, CNTK |
| Converters | $\mathcal{C}$ | $\approx 8$ | torch.onnx, tf2onnx, skl2onnx, onnxmltools, keras2onnx, paddle2onnx, mxnet.contrib.onnx, winmltools |
| Runtimes | $\mathcal{R}$ | $\approx 10$ | ORT, TensorRT, OpenVINO, CoreML, NNAPI, TVM, DirectML, WebNN, Caffe2, SNPE |
| Tools | $\mathcal{T}$ | $\approx 15$ | Netron, onnx-simplifier, onnxoptimizer, onnxruntime-tools, polygraphy, onnx.checker, onnx.shape_inference, ... |
| Model Zoo | $\mathcal{Z}$ | $\approx 180$ | Vision (ResNet, YOLO, EfficientNet), NLP (BERT, GPT-2, T5), Audio (Whisper), ... |
| Hardware Partners | $\mathcal{H}$ | $\approx 8$ | Intel, NVIDIA, AMD, Qualcomm, ARM, Apple, Xilinx, Huawei |

The **total ecosystem size** is $|\mathcal{E}|_{\text{total}} = \sum_{S \in \mathcal{E}} |S| \approx 231$ distinct components, making ONNX one of the largest interoperability ecosystems in machine learning.

In [ ]:
# Install required packages
!pip install onnx onnxruntime numpy matplotlib networkx -q

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import numpy as np

fig, ax = plt.subplots(figsize=(16, 10))
ax.set_xlim(0, 16)
ax.set_ylim(0, 10)
ax.axis('off')

categories = [
    ('Producers  P', 1.0, 8.0, 4.0, 1.2, '#FF6B6B',
     ['PyTorch', 'TensorFlow', 'JAX', 'scikit-learn', 'XGBoost', 'LightGBM']),
    ('Converters  C', 1.0, 6.2, 4.0, 1.2, '#FFA07A',
     ['torch.onnx', 'tf2onnx', 'skl2onnx', 'paddle2onnx', 'onnxmltools']),
    ('Runtimes  R', 9.5, 6.2, 5.5, 1.2, '#87CEEB',
     ['ORT', 'TensorRT', 'OpenVINO', 'CoreML', 'TVM', 'NNAPI']),
    ('Tools  T', 1.0, 4.0, 6.5, 1.2, '#98FB98',
     ['Netron', 'onnx-simplifier', 'onnxoptimizer', 'checker', 'shape_inference', 'polygraphy']),
    ('Model Zoo  Z', 1.0, 1.8, 6.5, 1.2, '#DDA0DD',
     ['ResNet', 'BERT', 'YOLOv5', 'GPT-2', 'EfficientNet', 'Whisper', 'CLIP']),
    ('Hardware  H', 9.5, 4.0, 5.5, 1.2, '#FFD700',
     ['Intel/OpenVINO', 'NVIDIA/TensorRT', 'AMD/ROCm', 'Qualcomm', 'ARM']),
]

for title, x, y, w, h, color, items in categories:
    rect = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.15',
                          facecolor=color, edgecolor='black', linewidth=1.5, alpha=0.85)
    ax.add_patch(rect)
    ax.text(x + 0.15, y + h - 0.25, title, fontsize=10, fontweight='bold', va='top')
    item_str = ', '.join(items)
    ax.text(x + 0.15, y + 0.25, item_str, fontsize=7.5, va='bottom', wrap=True,
            style='italic', color='#333333')

onnx_rect = FancyBboxPatch((6.0, 5.8), 2.8, 2.0, boxstyle='round,pad=0.2',
                            facecolor='#4ECDC4', edgecolor='black', linewidth=2.5)
ax.add_patch(onnx_rect)
ax.text(7.4, 7.1, 'ONNX', ha='center', va='center', fontsize=18, fontweight='bold')
ax.text(7.4, 6.6, 'Format + Spec', ha='center', va='center', fontsize=10, style='italic')
ax.text(7.4, 6.2, '.onnx  (protobuf)', ha='center', va='center', fontsize=8, color='#333')

arrows = [
    (5.0, 8.6, 6.0, 7.4, 'export'),
    (5.0, 6.8, 6.0, 6.8, 'convert'),
    (8.8, 6.8, 9.5, 6.8, 'execute'),
    (7.4, 5.8, 7.4, 5.2, 'transform'),
    (7.4, 3.0, 7.4, 5.8, 'provide'),
    (9.5, 5.2, 9.5, 4.0, 'accelerate'),
]

for x1, y1, x2, y2, label in arrows:
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#555', lw=2.0, connectionstyle='arc3,rad=0.1'))
    mx, my = (x1 + x2) / 2, (y1 + y2) / 2
    ax.text(mx + 0.15, my + 0.1, label, fontsize=7, color='#555', style='italic')

ax.set_title(r'Formal Taxonomy of the ONNX Ecosystem $\mathcal{E} = (\mathcal{P}, \mathcal{C}, \mathcal{R}, \mathcal{T}, \mathcal{Z}, \mathcal{H})$',
             fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

<a id='section-2'></a>
## Section 2: The Converter Pipeline Architecture

### The Full Pipeline

The ONNX ecosystem implements a multi-stage pipeline that transforms a trained model from its native framework representation into optimized inference on target hardware. Each stage performs a well-defined transformation with clear input/output contracts:

```
┌─────────────────────────────────────────────────────────────────────────────────────────────┐
│                        THE ONNX DEPLOYMENT PIPELINE                                        │
│                                                                                             │
│  ┌──────────┐    ┌──────────┐    ┌──────────┐    ┌───────────┐    ┌──────────┐    ┌──────┐ │
│  │ Training │    │ Exporter │    │   ONNX   │    │ Optimizer │    │ Runtime  │    │ HW   │ │
│  │ Framework│───▶│/Converter│───▶│  Model   │───▶│  /Tools   │───▶│ Engine   │───▶│Target│ │
│  │          │    │          │    │ (.onnx)  │    │           │    │          │    │      │ │
│  └──────────┘    └──────────┘    └──────────┘    └───────────┘    └──────────┘    └──────┘ │
│       │                │               │                │              │              │     │
│   PyTorch         torch.onnx      ModelProto      onnxoptimizer     ORT CPU       x86-64   │
│   TensorFlow      tf2onnx        GraphProto      onnx-simplifier   ORT CUDA      NVIDIA   │
│   sklearn         skl2onnx       NodeProto       ORT graph opt     TensorRT      Jetson   │
│   JAX             jax2onnx       TensorProto     quantization      OpenVINO      Intel    │
│   XGBoost         onnxmltools    OpSet v17+      pruning           CoreML        Apple    │
│                                                                                             │
└─────────────────────────────────────────────────────────────────────────────────────────────┘
```

### Formal Pipeline Definition

The pipeline $\Pi$ is a composition of transformations:

$$\Pi = \rho \circ \tau \circ \gamma \circ \kappa$$

where:
- $\kappa: \mathcal{M}_{\text{fw}} \to \mathcal{M}_{\text{onnx}}$ is the **conversion** function (framework model → ONNX)
- $\gamma: \mathcal{M}_{\text{onnx}} \to \mathcal{M}_{\text{onnx}}$ is the **graph optimization** function (identity-preserving rewrites)
- $\tau: \mathcal{M}_{\text{onnx}} \to \mathcal{M}_{\text{onnx}}^{q}$ is the optional **quantization/transformation** function
- $\rho: \mathcal{M}_{\text{onnx}} \to \mathcal{O}$ is the **runtime execution** function (model → inference outputs)

The correctness requirement is semantic equivalence at each stage:

$$\forall \mathbf{x}: \|\Pi(\mathbf{x}) - f_{\text{original}}(\mathbf{x})\|_\infty \leq \epsilon_{\text{tol}}$$

### Stage-by-Stage Breakdown

**Stage 1 — Export/Conversion ($\kappa$):** The exporter traces or scripts the framework model, mapping framework-specific operations to ONNX operators. This is the most error-prone stage because framework idioms (dynamic control flow, custom autograd functions, in-place operations) may not have direct ONNX equivalents. The exporter must resolve these mismatches while preserving numerical fidelity.

**Stage 2 — Graph Optimization ($\gamma$):** The optimizer applies rewrite rules to the ONNX graph: constant folding, redundant node elimination, operator fusion. These are semantics-preserving transformations: $\forall \mathbf{x}: \text{eval}(G, \mathbf{x}) = \text{eval}(\gamma(G), \mathbf{x})$. Tools like `onnxoptimizer` and `onnx-simplifier` operate at this level.

**Stage 3 — Quantization/Transformation ($\tau$):** Optional transformations that trade precision for speed. Post-training quantization maps FP32 weights and activations to INT8, reducing model size by $\sim4\times$ and enabling integer arithmetic. The tolerance $\epsilon_{\text{tol}}$ is typically bounded by task-level accuracy metrics rather than pointwise error.

**Stage 4 — Runtime Execution ($\rho$):** The runtime loads the optimized ONNX model, applies its own hardware-specific graph optimizations (EP-specific fusion), allocates memory, and executes inference. ORT's execution is deterministic given the same inputs on the same hardware.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(16, 6))
ax.set_xlim(0, 16)
ax.set_ylim(0, 6)
ax.axis('off')

stages = [
    (0.3, 2.0, 2.2, 2.0, 'Stage 1\nExport', '#FF6B6B', 'torch.onnx\ntf2onnx\nskl2onnx'),
    (3.2, 2.0, 2.2, 2.0, 'Stage 2\nValidate', '#FFA07A', 'onnx.checker\nshape_inference\nNetron'),
    (6.1, 2.0, 2.2, 2.0, 'Stage 3\nOptimize', '#98FB98', 'onnx-simplifier\nonnxoptimizer\nquantize'),
    (9.0, 2.0, 2.2, 2.0, 'Stage 4\nExecute', '#87CEEB', 'ONNX Runtime\nTensorRT\nOpenVINO'),
    (11.9, 2.0, 2.2, 2.0, 'Stage 5\nHardware', '#FFD700', 'CPU (x86/ARM)\nGPU (CUDA)\nNPU/DSP'),
]

for x, y, w, h, title, color, details in stages:
    rect = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.12',
                          facecolor=color, edgecolor='black', linewidth=1.5, alpha=0.85)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h - 0.35, title, ha='center', va='top',
            fontsize=10, fontweight='bold')
    ax.text(x + w/2, y + 0.4, details, ha='center', va='bottom',
            fontsize=7.5, style='italic', color='#333')

for i in range(len(stages) - 1):
    x1 = stages[i][0] + stages[i][2]
    x2 = stages[i+1][0]
    y_mid = stages[i][1] + stages[i][3] / 2
    ax.annotate('', xy=(x2, y_mid), xytext=(x1, y_mid),
                arrowprops=dict(arrowstyle='->', lw=2.5, color='#555'))

labels = [r'$\kappa$: convert', r'$\nu$: validate', r'$\gamma$: optimize',
          r'$\rho$: execute', '']
for i in range(len(stages) - 1):
    x1 = stages[i][0] + stages[i][2]
    x2 = stages[i+1][0]
    mx = (x1 + x2) / 2
    y_mid = stages[i][1] + stages[i][3] / 2
    ax.text(mx, y_mid + 0.3, labels[i], ha='center', fontsize=8, color='#333', style='italic')

fw_box = FancyBboxPatch((0.3, 0.3), 2.2, 1.0, boxstyle='round,pad=0.1',
                         facecolor='#E8E8E8', edgecolor='gray', linewidth=1)
ax.add_patch(fw_box)
ax.text(1.4, 0.8, 'Framework Model\n(PyTorch, TF, sklearn)', ha='center',
        va='center', fontsize=8)
ax.annotate('', xy=(1.4, 2.0), xytext=(1.4, 1.3),
            arrowprops=dict(arrowstyle='->', lw=1.5, color='gray'))

out_box = FancyBboxPatch((11.9, 0.3), 2.2, 1.0, boxstyle='round,pad=0.1',
                          facecolor='#E8E8E8', edgecolor='gray', linewidth=1)
ax.add_patch(out_box)
ax.text(13.0, 0.8, 'Inference Output\n(predictions)', ha='center',
        va='center', fontsize=8)
ax.annotate('', xy=(13.0, 0.3), xytext=(13.0, 2.0),
            arrowprops=dict(arrowstyle='->', lw=1.5, color='gray'))

ax.set_title('The ONNX Deployment Pipeline: Framework to Hardware',
             fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

<a id='section-3'></a>
## Section 3: Converters — Deep Dive

### The Conversion Function

Formally, a converter $\kappa_f$ for framework $f$ is a function:

$$\kappa_f: \mathcal{M}_f \times \mathcal{X}_{\text{sample}} \times \mathcal{O}_v \to \mathcal{M}_{\text{onnx}}$$

where $\mathcal{M}_f$ is the set of valid models in framework $f$, $\mathcal{X}_{\text{sample}}$ is a representative input (needed for tracing), and $\mathcal{O}_v$ is the target opset version. The converter must satisfy:

$$\forall \mathbf{x} \in \text{dom}(m): \|m(\mathbf{x}) - \text{eval}(\kappa_f(m, \mathbf{x}_s, v), \mathbf{x})\|_\infty \leq \epsilon_{\text{fp}}$$

### Converter Comparison

| Converter | Framework | Method | Dynamic Shapes | Custom Ops | Maturity |
|:---|:---|:---|:---:|:---:|:---:|
| `torch.onnx.export` | PyTorch | Tracing / TorchDynamo | Yes (v2.0+) | Via registration | ★★★★★ |
| `tf2onnx` | TensorFlow | Graph conversion | Yes | Partial | ★★★★☆ |
| `skl2onnx` | scikit-learn | Operator mapping | Limited | Via custom converters | ★★★★☆ |
| `paddle2onnx` | PaddlePaddle | Direct mapping | Yes | Yes | ★★★☆☆ |
| `onnxmltools` | Multiple ML | Wrapper | Varies | Varies | ★★★☆☆ |

### 3.1 torch.onnx — PyTorch to ONNX

The PyTorch exporter has evolved through several generations:

**Generation 1 (torch.onnx.export with tracing):** Uses `torch.jit.trace` to record operations on sample inputs. Limitations: cannot capture data-dependent control flow (`if tensor.sum() > 0`), and dynamic shapes require explicit specification via `dynamic_axes`.

**Generation 2 (TorchScript-based):** Uses `torch.jit.script` to compile Python code to TorchScript IR, then lowers to ONNX. Handles control flow but requires type annotations and TorchScript-compatible code.

**Generation 3 (TorchDynamo / torch.export, PyTorch 2.0+):** Uses Python bytecode analysis (`torch._dynamo`) to capture the full computation graph, including dynamic control flow. Produces `ExportedProgram` objects that can be serialized to ONNX with higher fidelity. This is the recommended path for new projects.

```python
# Generation 1: Tracing-based export
torch.onnx.export(model, dummy_input, "model.onnx", opset_version=17,
                  input_names=["input"], output_names=["output"],
                  dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}})

# Generation 3: TorchDynamo-based (PyTorch 2.1+)
export_output = torch.onnx.dynamo_export(model, dummy_input)
export_output.save("model_dynamo.onnx")
```

### 3.2 tf2onnx — TensorFlow to ONNX

`tf2onnx` converts TensorFlow SavedModels, frozen graphs, and Keras models to ONNX. It operates on the TensorFlow graph representation and maps TF operations to ONNX operators. Key considerations:

- **Op coverage:** ~95% of common TF ops are supported; exotic ops may need custom handling
- **Control flow:** `tf.cond` and `tf.while_loop` are mapped to ONNX `If` and `Loop` nodes
- **Shape inference:** TF's dynamic shapes are preserved via ONNX symbolic dimensions
- **Optimization:** tf2onnx applies its own graph optimizations during conversion

### 3.3 skl2onnx — scikit-learn to ONNX

The scikit-learn converter maps sklearn estimators to ONNX operators from the `ai.onnx.ml` domain. This domain includes operators like `LinearRegressor`, `TreeEnsembleClassifier`, and `SVMClassifier` that have no equivalent in the standard neural network operator set.

The converter handles the full sklearn pipeline API: `Pipeline`, `ColumnTransformer`, `FeatureUnion`, and custom transformers (with registration). The key challenge is that sklearn models are inherently non-neural — they include decision trees, SVMs, k-means, etc. — requiring a separate ONNX ML operator domain.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

converters = ['torch.onnx', 'tf2onnx', 'skl2onnx', 'paddle2onnx', 'onnxmltools']
metrics = ['Op Coverage', 'Dynamic Shapes', 'Custom Ops', 'Maturity', 'Community', 'Documentation']

scores = np.array([
    [0.95, 0.90, 0.80, 0.95, 0.95, 0.90],   # torch.onnx
    [0.90, 0.85, 0.60, 0.85, 0.80, 0.80],   # tf2onnx
    [0.85, 0.40, 0.70, 0.85, 0.75, 0.85],   # skl2onnx
    [0.75, 0.80, 0.65, 0.65, 0.50, 0.55],   # paddle2onnx
    [0.70, 0.50, 0.50, 0.70, 0.55, 0.60],   # onnxmltools
])

angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False).tolist()
angles += angles[:1]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#98FB98']

ax = axes[0]
for i, (conv, color) in enumerate(zip(converters, colors)):
    values = scores[i].tolist()
    values += values[:1]
    ax_polar = fig.add_axes(axes[0].get_position(), polar=True)
    ax_polar.fill(angles, values, alpha=0.15, color=color)
    ax_polar.plot(angles, values, 'o-', linewidth=2, label=conv, color=color, markersize=4)
    ax_polar.set_xticks(angles[:-1])
    ax_polar.set_xticklabels(metrics, fontsize=8)
    ax_polar.set_ylim(0, 1.0)
    ax_polar.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax_polar.set_yticklabels(['25%', '50%', '75%', '100%'], fontsize=7)
axes[0].axis('off')
ax_polar.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1), fontsize=8)
ax_polar.set_title('Converter Capability Radar', fontsize=12, fontweight='bold', pad=20)

ax2 = axes[1]
bar_width = 0.15
x = np.arange(len(metrics))
for i, (conv, color) in enumerate(zip(converters, colors)):
    ax2.barh(x + i * bar_width, scores[i], bar_width, label=conv, color=color, alpha=0.85)
ax2.set_yticks(x + bar_width * 2)
ax2.set_yticklabels(metrics, fontsize=9)
ax2.set_xlabel('Score (0-1)', fontsize=10)
ax2.set_title('Converter Feature Comparison', fontsize=12, fontweight='bold')
ax2.legend(fontsize=8, loc='lower right')
ax2.set_xlim(0, 1.1)
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

<a id='section-4'></a>
## Section 4: ONNX Model Zoo — Curated Model Collections

### Definition 4.1 (Model Zoo)

The ONNX Model Zoo $\mathcal{Z}$ is a curated repository of pre-trained models in ONNX format:

$$\mathcal{Z} = \{(m_i, \mathcal{D}_i, \mathcal{A}_i, \mathcal{S}_i) \mid i = 1, \ldots, |\mathcal{Z}|\}$$

where each entry consists of:
- $m_i \in \mathcal{M}_{\text{onnx}}$: the ONNX model (`.onnx` file)
- $\mathcal{D}_i$: the dataset it was trained on (ImageNet, COCO, SQuAD, ...)
- $\mathcal{A}_i$: the architecture family (ResNet, Transformer, ...)
- $\mathcal{S}_i$: metadata — accuracy metrics, model size, latency benchmarks

### Model Categories

The zoo is organized into functional categories, each serving different deployment scenarios:

| Category | Models | Typical Size | Key Architectures |
|:---|:---:|:---:|:---|
| **Image Classification** | ~40 | 5–300 MB | ResNet, VGG, EfficientNet, MobileNet, SqueezeNet, DenseNet |
| **Object Detection** | ~25 | 20–250 MB | YOLO (v3/v5/v8), SSD, Faster R-CNN, RetinaNet, DETR |
| **Semantic Segmentation** | ~10 | 50–400 MB | FCN, DeepLab, U-Net, PSPNet |
| **NLP / Text** | ~20 | 100 MB–2 GB | BERT, GPT-2, T5, RoBERTa, DistilBERT, XLNet |
| **Speech / Audio** | ~10 | 50–500 MB | Whisper, wav2vec 2.0, DeepSpeech |
| **Generative** | ~8 | 500 MB–5 GB | Stable Diffusion (UNet/VAE), StyleGAN |
| **Super Resolution** | ~5 | 5–100 MB | SRCNN, EDSR, Real-ESRGAN |
| **Body/Face/Hand** | ~15 | 10–200 MB | ArcFace, MediaPipe, OpenPose |

### Usage Patterns

Model Zoo models serve several critical roles in the ecosystem:

**1. Baseline Benchmarking:** Teams use zoo models as performance baselines when evaluating runtimes or hardware. Running ResNet-50 from the zoo on ORT vs. TensorRT provides an apples-to-apples comparison without confounding variables from export quality.

**2. Transfer Learning:** Pre-trained vision and NLP models serve as feature extractors. The ONNX format preserves the learned representations, enabling fine-tuning in frameworks that support ONNX import or feature extraction via inference.

**3. Deployment Templates:** Zoo models demonstrate best practices for input/output specification, dynamic axis naming, opset selection, and metadata annotation. They serve as reference implementations for teams building their own export pipelines.

**4. Compatibility Testing:** Hardware vendors use the zoo to validate their ONNX runtime implementations. A model that runs correctly on the reference ONNX Runtime but fails on a vendor's runtime reveals a bug in the vendor's implementation.

### The Model Zoo Access Protocol

```
┌──────────────┐     ┌─────────────┐     ┌──────────────┐     ┌──────────────┐
│  onnx.hub    │────▶│   GitHub     │────▶│  .onnx file  │────▶│  ORT Session │
│  .load()     │     │  Releases    │     │  (download)  │     │  .run()      │
└──────────────┘     └─────────────┘     └──────────────┘     └──────────────┘
       │                                                             │
       └─── list_models()                                            └─── predictions
            get_model_info()
```

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

categories = ['Image\nClassification', 'Object\nDetection', 'Semantic\nSegmentation',
              'NLP/Text', 'Speech\n/Audio', 'Generative', 'Super\nResolution', 'Body/Face\n/Hand']
model_counts = [40, 25, 10, 20, 10, 8, 5, 15]
avg_sizes_mb = [80, 120, 200, 500, 250, 2000, 30, 80]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

colors = plt.cm.Set3(np.linspace(0, 1, len(categories)))

ax1 = axes[0]
bars = ax1.bar(range(len(categories)), model_counts, color=colors, edgecolor='black', linewidth=0.8)
ax1.set_xticks(range(len(categories)))
ax1.set_xticklabels(categories, fontsize=8, rotation=0)
ax1.set_ylabel('Number of Models', fontsize=10)
ax1.set_title('ONNX Model Zoo: Models per Category', fontsize=11, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)
for bar, count in zip(bars, model_counts):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             str(count), ha='center', va='bottom', fontsize=9, fontweight='bold')

ax2 = axes[1]
wedges, texts, autotexts = ax2.pie(model_counts, labels=categories, autopct='%1.0f%%',
                                    colors=colors, textprops={'fontsize': 7.5},
                                    pctdistance=0.8)
ax2.set_title('Distribution by Category', fontsize=11, fontweight='bold')

ax3 = axes[2]
ax3.barh(range(len(categories)), avg_sizes_mb, color=colors, edgecolor='black', linewidth=0.8)
ax3.set_yticks(range(len(categories)))
ax3.set_yticklabels(categories, fontsize=8)
ax3.set_xlabel('Average Model Size (MB)', fontsize=10)
ax3.set_title('Average Model Size by Category', fontsize=11, fontweight='bold')
ax3.set_xscale('log')
ax3.grid(axis='x', alpha=0.3)
for i, size in enumerate(avg_sizes_mb):
    ax3.text(size + size*0.1, i, f'{size} MB', va='center', fontsize=8)

plt.tight_layout()
plt.show()

print(f'Total models in zoo: {sum(model_counts)}')
print(f'Average size across all categories: {np.mean(avg_sizes_mb):.0f} MB')
print(f'Size range: {min(avg_sizes_mb)} MB - {max(avg_sizes_mb)} MB')

<a id='section-5'></a>
## Section 5: Runtime Landscape — ORT, TensorRT, OpenVINO

### Definition 5.1 (Runtime)

An ONNX runtime $r \in \mathcal{R}$ is a function:

$$r: \mathcal{M}_{\text{onnx}} \times \mathcal{X} \to \mathcal{Y}$$

that takes an ONNX model and input tensors, and produces output tensors. A runtime must satisfy the ONNX specification semantics for all operators it claims to support.

### The Runtime Landscape

The three dominant ONNX runtimes represent fundamentally different design philosophies:

**ONNX Runtime (ORT)** is Microsoft's reference implementation and the most widely deployed ONNX runtime. Its distinguishing feature is the **Execution Provider (EP)** architecture: a plugin system that allows hardware vendors to contribute optimized operator implementations without modifying the core runtime. ORT ships with EPs for CPU (OpenMP/MKL), CUDA, TensorRT, OpenVINO, DirectML, CoreML, NNAPI, and WebNN. This makes ORT the most portable runtime — a single API for deployment across all major platforms.

**TensorRT** is NVIDIA's deep learning inference optimizer and runtime. Unlike ORT, TensorRT is hardware-specific: it targets NVIDIA GPUs exclusively. Its advantage is deep optimization: layer fusion, precision calibration (FP16/INT8), kernel auto-tuning, and memory optimization produce inference speeds that are typically 2-5× faster than generic GPU execution. TensorRT can consume ONNX models directly via its parser, or be accessed through ORT's TensorRT EP.

**OpenVINO** is Intel's inference toolkit targeting Intel CPUs, GPUs, VPUs (Movidius), and FPGAs. It performs aggressive CPU-specific optimizations: vectorization for AVX-512, operator fusion for Intel architectures, and multi-stream inference for throughput maximization. OpenVINO can import ONNX models directly or be accessed through ORT's OpenVINO EP.

### Runtime Feature Matrix

| Feature | ORT | TensorRT | OpenVINO |
|:---|:---:|:---:|:---:|
| **Target Hardware** | Cross-platform | NVIDIA GPU only | Intel HW |
| **ONNX OpSet Support** | Latest (20+) | Most ops (17+) | Most ops (17+) |
| **FP16 Inference** | Via EP | Native | Supported |
| **INT8 Quantization** | Built-in tools | Calibration-based | POT/NNCF |
| **Dynamic Shapes** | Full support | Limited | Supported |
| **Multi-threading** | OpenMP | CUDA streams | TBB |
| **Graph Optimization** | 3 levels | Extensive | Extensive |
| **Model Caching** | Session options | Engine serialization | Compiled model |
| **Language Bindings** | Python, C++, C#, Java, JS | Python, C++ | Python, C++ |
| **Edge Deployment** | ONNX Runtime Mobile | TensorRT Lite | OpenVINO Lite |

### Performance Characteristics

The latency $L$ of a runtime $r$ on model $m$ depends on:

$$L(r, m) = T_{\text{load}} + T_{\text{optimize}} + T_{\text{alloc}} + T_{\text{infer}}$$

where:
- $T_{\text{load}}$: model deserialization time
- $T_{\text{optimize}}$: graph optimization time (can be cached)
- $T_{\text{alloc}}$: memory allocation for activations
- $T_{\text{infer}}$: actual computation time

For serving workloads, $T_{\text{infer}}$ dominates. For serverless/cold-start scenarios, $T_{\text{load}} + T_{\text{optimize}}$ may dominate. This tradeoff shapes runtime selection:

$$\text{ORT:} \quad T_{\text{load}} \text{ (fast)} + T_{\text{infer}} \text{ (moderate)}$$
$$\text{TensorRT:} \quad T_{\text{load}} \text{ (slow, builds engine)} + T_{\text{infer}} \text{ (fastest on NVIDIA)}$$
$$\text{OpenVINO:} \quad T_{\text{load}} \text{ (moderate)} + T_{\text{infer}} \text{ (fastest on Intel)}$$

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

runtimes = ['ONNX Runtime', 'TensorRT', 'OpenVINO', 'CoreML', 'TVM', 'NNAPI', 'DirectML']
features = ['Cross-Platform', 'GPU Accel', 'INT8 Quant', 'Dynamic Shapes',
            'Graph Optim', 'Edge Deploy', 'Op Coverage', 'Community']

scores = np.array([
    [1.0, 0.9, 0.8, 1.0, 0.9, 0.7, 1.0, 1.0],   # ORT
    [0.2, 1.0, 1.0, 0.5, 1.0, 0.6, 0.8, 0.8],   # TensorRT
    [0.4, 0.7, 0.9, 0.8, 0.9, 0.8, 0.85, 0.7],  # OpenVINO
    [0.2, 0.6, 0.7, 0.7, 0.7, 0.9, 0.6, 0.5],   # CoreML
    [0.9, 0.8, 0.7, 0.8, 0.8, 0.7, 0.7, 0.6],   # TVM
    [0.2, 0.5, 0.6, 0.5, 0.5, 0.9, 0.5, 0.3],   # NNAPI
    [0.3, 0.8, 0.4, 0.7, 0.6, 0.3, 0.6, 0.4],   # DirectML
])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7), gridspec_kw={'width_ratios': [2, 1]})

im = ax1.imshow(scores, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
ax1.set_xticks(range(len(features)))
ax1.set_yticks(range(len(runtimes)))
ax1.set_xticklabels(features, rotation=45, ha='right', fontsize=9)
ax1.set_yticklabels(runtimes, fontsize=10)

for i in range(len(runtimes)):
    for j in range(len(features)):
        val = scores[i, j]
        color = 'white' if val < 0.5 else 'black'
        ax1.text(j, i, f'{val:.1f}', ha='center', va='center', fontsize=9,
                color=color, fontweight='bold')

plt.colorbar(im, ax=ax1, fraction=0.02, pad=0.04, label='Capability Score')
ax1.set_title('Runtime Feature Matrix', fontsize=13, fontweight='bold')

avg_scores = scores.mean(axis=1)
sorted_idx = np.argsort(avg_scores)[::-1]
colors_bar = plt.cm.Set2(np.linspace(0, 1, len(runtimes)))

ax2.barh(range(len(runtimes)), avg_scores[sorted_idx],
         color=[colors_bar[i] for i in sorted_idx], edgecolor='black', linewidth=0.8)
ax2.set_yticks(range(len(runtimes)))
ax2.set_yticklabels([runtimes[i] for i in sorted_idx], fontsize=10)
ax2.set_xlabel('Average Score', fontsize=10)
ax2.set_title('Overall Ranking', fontsize=13, fontweight='bold')
ax2.set_xlim(0, 1.1)
ax2.grid(axis='x', alpha=0.3)

for i, idx in enumerate(sorted_idx):
    ax2.text(avg_scores[idx] + 0.02, i, f'{avg_scores[idx]:.2f}',
             va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

<a id='section-6'></a>
## Section 6: Execution Provider Architecture

### The EP Abstraction

ONNX Runtime's **Execution Provider (EP)** system is its most architecturally significant feature. An EP is a hardware-specific backend that implements a subset of ONNX operators with optimized kernels:

$$\text{EP}: \mathcal{O}_{\text{supported}} \subseteq \mathcal{O}_{\text{onnx}} \to \text{KernelImpl}$$

When ORT loads a model, it **partitions** the graph across available EPs using a priority-based assignment algorithm:

```
┌─────────────────────────────────────────────────────────────────────┐
│                    ONNX Runtime Session                             │
│                                                                     │
│  ┌──────────────────────────────────────────────────────────────┐  │
│  │                    Graph Partitioner                          │  │
│  │                                                              │  │
│  │  For each node n in topological order:                       │  │
│  │    for each EP in priority order:                            │  │
│  │      if EP.supports(n.op_type, n.attributes):                │  │
│  │        assign n → EP                                         │  │
│  │        break                                                 │  │
│  │                                                              │  │
│  │  Result: Graph partitioned into EP-specific subgraphs        │  │
│  └──────────────────────────────────────────────────────────────┘  │
│                                                                     │
│  ┌────────────┐  ┌────────────┐  ┌────────────┐  ┌────────────┐  │
│  │ TensorRT   │  │  CUDA EP   │  │ OpenVINO   │  │  CPU EP    │  │
│  │    EP      │  │            │  │    EP      │  │ (fallback) │  │
│  │            │  │            │  │            │  │            │  │
│  │ Conv, MatMul│ │ Relu, BN  │  │ Custom ops │  │ All ops    │  │
│  │ Attention  │  │ Softmax   │  │            │  │ (default)  │  │
│  └─────┬──────┘  └─────┬──────┘  └─────┬──────┘  └─────┬──────┘  │
│        │               │               │               │          │
│        ▼               ▼               ▼               ▼          │
│  ┌──────────┐   ┌──────────┐   ┌──────────┐   ┌──────────┐      │
│  │ NVIDIA   │   │ NVIDIA   │   │  Intel   │   │ x86/ARM  │      │
│  │ GPU      │   │ GPU      │   │ CPU/GPU  │   │ CPU      │      │
│  └──────────┘   └──────────┘   └──────────┘   └──────────┘      │
└─────────────────────────────────────────────────────────────────────┘
```

### The Partitioning Algorithm

The graph partitioning is a node coloring problem. Given EP priority list $[e_1, e_2, \ldots, e_k]$ where $e_k$ is the CPU fallback:

$$\text{color}(v) = e_j \quad \text{where } j = \min\{i \mid e_i.\text{supports}(v)\}$$

This greedy assignment maximizes utilization of the highest-priority (most accelerated) EP. The fallback EP (CPU) must support all ONNX operators, guaranteeing that every node is assigned.

### Data Transfer Costs

When adjacent nodes are assigned to different EPs, data must be transferred between devices. The total execution cost includes these transfer penalties:

$$T_{\text{total}} = \sum_{v \in V} T_{\text{compute}}(v) + \sum_{(u,v) \in E_{\text{cross-EP}}} T_{\text{transfer}}(u, v)$$

where $E_{\text{cross-EP}} = \{(u,v) \in E \mid \text{color}(u) \neq \text{color}(v)\}$. Excessive graph fragmentation (many EP transitions) can negate the benefits of hardware acceleration due to transfer overhead. This is why runtimes try to **fuse** subgraphs assigned to the same EP into single kernel calls.

### Available EPs in ORT

| EP | Hardware | Key Optimizations |
|:---|:---|:---|
| **CPUExecutionProvider** | x86, ARM | MKL-DNN, OpenMP parallelism |
| **CUDAExecutionProvider** | NVIDIA GPU | cuDNN, cuBLAS kernels |
| **TensorrtExecutionProvider** | NVIDIA GPU | Layer fusion, FP16/INT8, kernel tuning |
| **OpenVINOExecutionProvider** | Intel CPU/GPU/VPU | AVX-512, nGraph optimizations |
| **DirectMLExecutionProvider** | DirectX 12 GPU | Windows ML acceleration |
| **CoreMLExecutionProvider** | Apple Neural Engine | ANE delegation |
| **NnapiExecutionProvider** | Android NPU/DSP | Mobile neural accelerators |
| **WebNNExecutionProvider** | Browser | WebNN API delegation |

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import numpy as np

fig, ax = plt.subplots(figsize=(14, 9))
ax.set_xlim(0, 14)
ax.set_ylim(0, 9)
ax.axis('off')

title_box = FancyBboxPatch((3, 7.8), 8, 0.8, boxstyle='round,pad=0.15',
                            facecolor='#2C3E50', edgecolor='black', linewidth=2)
ax.add_patch(title_box)
ax.text(7, 8.2, 'ONNX Runtime Session', ha='center', va='center',
        fontsize=14, fontweight='bold', color='white')

model_box = FancyBboxPatch((0.5, 7.0), 3.0, 0.6, boxstyle='round,pad=0.1',
                            facecolor='#E8E8E8', edgecolor='gray', linewidth=1)
ax.add_patch(model_box)
ax.text(2.0, 7.3, 'ONNX Model (.onnx)', ha='center', va='center', fontsize=9)
ax.annotate('', xy=(3.5, 8.0), xytext=(3.0, 7.5),
            arrowprops=dict(arrowstyle='->', lw=1.5, color='gray'))

part_box = FancyBboxPatch((4, 6.5), 6, 0.8, boxstyle='round,pad=0.1',
                           facecolor='#ECF0F1', edgecolor='#2C3E50', linewidth=1.5)
ax.add_patch(part_box)
ax.text(7, 6.9, 'Graph Partitioner (priority-based node assignment)', ha='center',
        va='center', fontsize=9, fontweight='bold')
ax.annotate('', xy=(7, 6.5), xytext=(7, 7.8),
            arrowprops=dict(arrowstyle='->', lw=1.5, color='#2C3E50'))

eps = [
    ('TensorRT EP', '#76D7C4', 'Conv, MatMul\nAttention, Gemm', 'Priority 1'),
    ('CUDA EP', '#85C1E9', 'Relu, BN, Softmax\nElementwise ops', 'Priority 2'),
    ('OpenVINO EP', '#F9E79F', 'CPU-optimized\nsubgraphs', 'Priority 3'),
    ('CPU EP', '#F5B7B1', 'ALL operators\n(fallback)', 'Fallback'),
]

for i, (name, color, ops, priority) in enumerate(eps):
    x = 0.5 + i * 3.3
    rect = FancyBboxPatch((x, 4.0), 2.8, 2.0, boxstyle='round,pad=0.1',
                          facecolor=color, edgecolor='black', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x + 1.4, 5.6, name, ha='center', va='center', fontsize=9, fontweight='bold')
    ax.text(x + 1.4, 4.9, ops, ha='center', va='center', fontsize=7.5, style='italic')
    ax.text(x + 1.4, 4.25, priority, ha='center', va='center', fontsize=7.5,
            color='#555', fontweight='bold')
    ax.annotate('', xy=(x + 1.4, 6.0), xytext=(x + 1.4, 6.5),
                arrowprops=dict(arrowstyle='->', lw=1.5, color='#555'))

hw_labels = ['NVIDIA GPU', 'NVIDIA GPU', 'Intel CPU/GPU', 'x86/ARM CPU']
hw_colors = ['#27AE60', '#27AE60', '#2980B9', '#E74C3C']
for i, (label, color) in enumerate(zip(hw_labels, hw_colors)):
    x = 0.5 + i * 3.3
    rect = FancyBboxPatch((x, 2.5), 2.8, 1.0, boxstyle='round,pad=0.1',
                          facecolor=color, edgecolor='black', linewidth=1.5, alpha=0.7)
    ax.add_patch(rect)
    ax.text(x + 1.4, 3.0, label, ha='center', va='center', fontsize=9,
            fontweight='bold', color='white')
    ax.annotate('', xy=(x + 1.4, 3.5), xytext=(x + 1.4, 4.0),
                arrowprops=dict(arrowstyle='->', lw=1.5, color='#555'))

out_box = FancyBboxPatch((4, 1.0), 6, 0.8, boxstyle='round,pad=0.1',
                          facecolor='#D5F5E3', edgecolor='#27AE60', linewidth=1.5)
ax.add_patch(out_box)
ax.text(7, 1.4, 'Inference Output (predictions)', ha='center', va='center',
        fontsize=10, fontweight='bold')

for i in range(4):
    x = 0.5 + i * 3.3 + 1.4
    ax.annotate('', xy=(7, 1.8), xytext=(x, 2.5),
                arrowprops=dict(arrowstyle='->', lw=1.0, color='gray', ls='--'))

ax.set_title('ONNX Runtime Execution Provider Architecture',
             fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

<a id='section-7'></a>
## Section 7: Tools Ecosystem — Visualization, Validation, Optimization

### The Tool Classification

The ONNX tools ecosystem $\mathcal{T}$ can be partitioned into four functional categories:

$$\mathcal{T} = \mathcal{T}_{\text{viz}} \cup \mathcal{T}_{\text{val}} \cup \mathcal{T}_{\text{opt}} \cup \mathcal{T}_{\text{dbg}}$$

where:
- $\mathcal{T}_{\text{viz}}$ = **Visualization tools** — for human inspection of graph structure
- $\mathcal{T}_{\text{val}}$ = **Validation tools** — for checking model correctness
- $\mathcal{T}_{\text{opt}}$ = **Optimization tools** — for graph rewrites and compression
- $\mathcal{T}_{\text{dbg}}$ = **Debugging tools** — for diagnosing inference issues

### 7.1 Visualization — Netron

**Netron** is the de-facto standard for visual model inspection. It renders ONNX graphs as interactive node-link diagrams, showing operator types, tensor shapes, data types, and attributes. Netron operates as:

- A standalone desktop application (Electron-based)
- A web application (netron.app)
- A Python package (`import netron; netron.start('model.onnx')`)
- A VS Code extension

For each node $v_i$ in the graph, Netron displays:
- **Op type**: The ONNX operator (Conv, MatMul, Relu, ...)
- **Inputs/Outputs**: Tensor names with inferred shapes and types
- **Attributes**: Operator parameters (kernel_shape, strides, padding, ...)
- **Initializer values**: Weight tensor statistics (shape, dtype, min/max)

Netron is essential for debugging export issues: when a converted model produces incorrect results, visual inspection often reveals missing operators, shape mismatches, or incorrect attribute values that are invisible in serialized protobuf.

### 7.2 Validation — onnx.checker and shape_inference

The ONNX Python package includes two critical validation tools:

**onnx.checker.check_model(model):** Validates the structural integrity of an ONNX model against the specification. It verifies:
- All referenced operator types exist in the declared opset
- Input/output tensor names form valid connections
- Attribute types match operator specifications
- The graph is a valid DAG (no cycles)

**onnx.shape_inference.infer_shapes(model):** Propagates shape and type information through the graph, computing output shapes from input shapes and operator semantics. This is the ONNX equivalent of type inference in a programming language.

### 7.3 Optimization — onnx-simplifier and onnxoptimizer

**onnx-simplifier** reduces model complexity by evaluating constant subexpressions and simplifying redundant patterns. It uses ONNX Runtime as an oracle: it runs constant subgraphs, captures the outputs, and replaces those subgraphs with constant tensors. The result is a graph with fewer nodes and no "dead" computation.

**onnxoptimizer** applies a library of named rewrite passes: `eliminate_identity`, `fuse_bn_into_conv`, `fuse_matmul_add_bias_into_gemm`, etc. Each pass is a local graph transformation rule of the form:

$$\text{pass}: G \to G' \quad \text{where} \quad \forall \mathbf{x}: \text{eval}(G, \mathbf{x}) = \text{eval}(G', \mathbf{x})$$

### 7.4 Debugging — polygraphy and onnxruntime-tools

**polygraphy** (NVIDIA) is a toolkit for debugging deep learning inference, including ONNX models. It can:
- Compare outputs across runtimes (ONNX Runtime vs TensorRT)
- Identify the first layer where outputs diverge
- Generate reduced test cases for bug reporting
- Inspect intermediate tensor values

### Summary Table

| Tool | Category | Purpose | Input | Output |
|:---|:---:|:---|:---|:---|
| Netron | $\mathcal{T}_{\text{viz}}$ | Visual graph inspection | .onnx file | Interactive diagram |
| onnx.checker | $\mathcal{T}_{\text{val}}$ | Structural validation | ModelProto | Pass/Fail + errors |
| shape_inference | $\mathcal{T}_{\text{val}}$ | Type/shape propagation | ModelProto | Annotated ModelProto |
| onnx-simplifier | $\mathcal{T}_{\text{opt}}$ | Constant folding + simplification | .onnx file | Simplified .onnx |
| onnxoptimizer | $\mathcal{T}_{\text{opt}}$ | Named rewrite passes | ModelProto | Optimized ModelProto |
| polygraphy | $\mathcal{T}_{\text{dbg}}$ | Cross-runtime comparison | .onnx + inputs | Divergence report |

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

G = nx.DiGraph()

tool_categories = {
    'Visualization': ['Netron', 'Netron.app\n(web)', 'VS Code\nExt'],
    'Validation': ['onnx.checker', 'shape_inference', 'onnx.version\n_converter'],
    'Optimization': ['onnx-simplifier', 'onnxoptimizer', 'ORT Graph\nOpt'],
    'Debugging': ['polygraphy', 'onnxruntime\n-tools', 'onnx.numpy\n_helper'],
}

color_map = {
    'Visualization': '#FF6B6B',
    'Validation': '#4ECDC4',
    'Optimization': '#45B7D1',
    'Debugging': '#FFA07A',
}

G.add_node('ONNX\nModel', category='center')

for cat, tools in tool_categories.items():
    G.add_node(cat, category='category')
    G.add_edge('ONNX\nModel', cat)
    for tool in tools:
        G.add_node(tool, category=cat)
        G.add_edge(cat, tool)

pos = {}
pos['ONNX\nModel'] = (0, 0)

cat_positions = {
    'Visualization': (-3, 2),
    'Validation': (3, 2),
    'Optimization': (-3, -2),
    'Debugging': (3, -2),
}

for cat, (cx, cy) in cat_positions.items():
    pos[cat] = (cx, cy)
    tools = tool_categories[cat]
    for j, tool in enumerate(tools):
        angle = np.pi/4 * (j - 1)
        dx = np.sign(cx) * 2.0
        dy = (j - 1) * 1.2
        pos[tool] = (cx + dx, cy + dy)

fig, ax = plt.subplots(figsize=(14, 10))

node_colors = []
node_sizes = []
for node in G.nodes():
    if node == 'ONNX\nModel':
        node_colors.append('#FFD700')
        node_sizes.append(3000)
    elif node in tool_categories:
        node_colors.append(color_map[node])
        node_sizes.append(2000)
    else:
        cat = G.nodes[node].get('category', 'center')
        node_colors.append(color_map.get(cat, '#E8E8E8'))
        node_sizes.append(1200)

nx.draw(G, pos, ax=ax, with_labels=True, node_color=node_colors,
        node_size=node_sizes, font_size=7.5, font_weight='bold',
        arrows=True, arrowsize=15, edge_color='#999',
        connectionstyle='arc3,rad=0.1', alpha=0.9)

import matplotlib.patches as mpatches
legend_elements = [
    mpatches.Patch(color='#FFD700', label='ONNX Model (center)'),
    mpatches.Patch(color='#FF6B6B', label='Visualization tools'),
    mpatches.Patch(color='#4ECDC4', label='Validation tools'),
    mpatches.Patch(color='#45B7D1', label='Optimization tools'),
    mpatches.Patch(color='#FFA07A', label='Debugging tools'),
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=9)
ax.set_title(r'ONNX Tools Ecosystem: $\mathcal{T} = \mathcal{T}_{viz} \cup \mathcal{T}_{val} \cup \mathcal{T}_{opt} \cup \mathcal{T}_{dbg}$',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

<a id='section-8'></a>
## Section 8: Ecosystem Growth Timeline

### Historical Development

The ONNX ecosystem has grown from a simple interchange format (2017) to a comprehensive deployment platform encompassing hundreds of components. This growth follows a pattern common to successful open-source ecosystems: initial specification → reference implementation → community adoption → hardware vendor engagement → enterprise deployment.

```
ONNX Ecosystem Timeline
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
2017 Q3 ─── ONNX announced by Facebook + Microsoft
            ├── Initial spec: ~80 operators (OpSet 1-5)
            ├── Caffe2 and PyTorch exporters
            └── Goal: "train once, deploy anywhere"

2018 Q1 ─── ONNX Runtime (ORT) v0.1 released by Microsoft
            ├── CPU execution provider only
            ├── tf2onnx converter launched
            └── Model Zoo: ~20 models (vision only)

2018 Q4 ─── ONNX 1.4: OpSet 9
            ├── CUDA EP added to ORT
            ├── Netron adds ONNX support
            └── skl2onnx released for scikit-learn

2019 Q2 ─── ONNX joins Linux Foundation (LF AI)
            ├── TensorRT EP in ORT
            ├── OpenVINO EP in ORT
            ├── Model Zoo: ~60 models
            └── onnxoptimizer released

2020 Q1 ─── ONNX 1.7: OpSet 12-13
            ├── Dynamic shapes mature
            ├── Training support (ONNX-Training spec)
            ├── onnx-simplifier released
            └── Model Zoo: ~100 models

2021 Q1 ─── ONNX graduates to LF AI & Data
            ├── ORT 1.8: WebNN EP, NNAPI EP
            ├── Model Zoo: ~130 models (adds NLP)
            └── Quantization tools in ORT

2022 Q2 ─── PyTorch 2.0: torch.onnx improvements
            ├── ONNX 1.12: OpSet 17
            ├── TorchDynamo-based export
            ├── ORT Mobile for edge deployment
            └── Model Zoo: ~150 models

2023 Q2 ─── ONNX 1.14: OpSet 20
            ├── Transformer operator patterns mature
            ├── LLM-related operators
            ├── ORT 1.16: improved EP ecosystem
            └── Model Zoo: ~180 models (adds generative)

2024 Q1 ─── ONNX 1.16: OpSet 21
            ├── Further LLM optimizations
            ├── Quantization tooling matures
            └── Ecosystem: 230+ components
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
```

### Growth Metrics

The ecosystem's growth can be quantified across multiple dimensions:

| Year | OpSet | Operators | Model Zoo | EPs | Converters | GitHub Stars (ORT) |
|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| 2017 | 1-5 | ~80 | 5 | 1 | 2 | — |
| 2018 | 6-9 | ~130 | 20 | 3 | 4 | ~1k |
| 2019 | 10-11 | ~150 | 60 | 5 | 5 | ~3k |
| 2020 | 12-13 | ~170 | 100 | 7 | 6 | ~5k |
| 2021 | 14-15 | ~180 | 130 | 9 | 7 | ~7k |
| 2022 | 16-17 | ~185 | 150 | 10 | 8 | ~10k |
| 2023 | 18-20 | ~195 | 180 | 11 | 8 | ~13k |
| 2024 | 21 | ~200 | 180+ | 12 | 8 | ~15k |

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

years = [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
operators = [80, 130, 150, 170, 180, 185, 195, 200]
model_zoo = [5, 20, 60, 100, 130, 150, 180, 180]
eps = [1, 3, 5, 7, 9, 10, 11, 12]
github_stars_k = [0, 1, 3, 5, 7, 10, 13, 15]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ax1 = axes[0, 0]
ax1.plot(years, operators, 'o-', color='#FF6B6B', linewidth=2.5, markersize=8)
ax1.fill_between(years, operators, alpha=0.15, color='#FF6B6B')
ax1.set_ylabel('Count', fontsize=10)
ax1.set_title('ONNX Standard Operators', fontsize=12, fontweight='bold')
ax1.grid(alpha=0.3)
ax1.set_xlim(2016.5, 2024.5)
for x, y in zip(years, operators):
    ax1.annotate(str(y), (x, y), textcoords='offset points', xytext=(0, 10),
                ha='center', fontsize=8)

ax2 = axes[0, 1]
ax2.plot(years, model_zoo, 's-', color='#4ECDC4', linewidth=2.5, markersize=8)
ax2.fill_between(years, model_zoo, alpha=0.15, color='#4ECDC4')
ax2.set_ylabel('Count', fontsize=10)
ax2.set_title('Model Zoo Size', fontsize=12, fontweight='bold')
ax2.grid(alpha=0.3)
ax2.set_xlim(2016.5, 2024.5)
for x, y in zip(years, model_zoo):
    ax2.annotate(str(y), (x, y), textcoords='offset points', xytext=(0, 10),
                ha='center', fontsize=8)

ax3 = axes[1, 0]
ax3.bar(years, eps, color='#45B7D1', edgecolor='black', linewidth=0.8, width=0.6)
ax3.set_ylabel('Count', fontsize=10)
ax3.set_xlabel('Year', fontsize=10)
ax3.set_title('Execution Providers in ORT', fontsize=12, fontweight='bold')
ax3.grid(axis='y', alpha=0.3)
for x, y in zip(years, eps):
    ax3.text(x, y + 0.2, str(y), ha='center', fontsize=9, fontweight='bold')

ax4 = axes[1, 1]
ax4.plot(years, github_stars_k, 'D-', color='#FFA07A', linewidth=2.5, markersize=8)
ax4.fill_between(years, github_stars_k, alpha=0.15, color='#FFA07A')
ax4.set_ylabel('Stars (thousands)', fontsize=10)
ax4.set_xlabel('Year', fontsize=10)
ax4.set_title('ORT GitHub Stars', fontsize=12, fontweight='bold')
ax4.grid(alpha=0.3)
ax4.set_xlim(2016.5, 2024.5)
for x, y in zip(years, github_stars_k):
    ax4.annotate(f'{y}k', (x, y), textcoords='offset points', xytext=(0, 10),
                ha='center', fontsize=8)

plt.suptitle('ONNX Ecosystem Growth (2017-2024)', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

<a id='section-9'></a>
## Section 9: The Full Ecosystem Interaction Graph

### Graph-Theoretic Model

The complete ONNX ecosystem can be modeled as a directed graph $G_{\mathcal{E}} = (V, E)$ where:

$$V = \mathcal{P} \cup \mathcal{C} \cup \{\text{ONNX}\} \cup \mathcal{T} \cup \mathcal{R} \cup \mathcal{H} \cup \mathcal{Z}$$

and edges $E$ represent data flow, API calls, or plugin relationships. This graph has several important properties:

**Property 9.1 (Hub structure):** The ONNX format node has the highest betweenness centrality in $G_{\mathcal{E}}$:

$$\text{betweenness}(\text{ONNX}) = \max_{v \in V} \text{betweenness}(v)$$

This confirms ONNX's role as the "hub" of a hub-and-spoke architecture.

**Property 9.2 (Layered structure):** The graph admits a layered decomposition:

$$V = L_1 \cup L_2 \cup L_3 \cup L_4 \cup L_5$$

where $L_1 = \mathcal{P}$ (producers), $L_2 = \mathcal{C}$ (converters), $L_3 = \{\text{ONNX}\} \cup \mathcal{T}$ (format + tools), $L_4 = \mathcal{R}$ (runtimes), $L_5 = \mathcal{H}$ (hardware). All edges go from lower to higher layers (with tools having self-loops on $L_3$).

**Property 9.3 (Modularity):** Adding a new framework $f_{N+1}$ requires only one new converter, not changes to any runtime. Similarly, adding a new runtime $r_{M+1}$ requires only an ONNX parser, not changes to any framework. This is the modularity guarantee of the hub architecture.

The following visualization renders the full ecosystem graph using NetworkX, with node positions arranged in the layered layout and edge weights proportional to interaction frequency.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
import numpy as np

G = nx.DiGraph()

layers = {
    'Frameworks': {
        'nodes': ['PyTorch', 'TensorFlow', 'JAX', 'sklearn', 'XGBoost', 'LightGBM'],
        'color': '#FF6B6B', 'y': 5
    },
    'Converters': {
        'nodes': ['torch.onnx', 'tf2onnx', 'jax2onnx', 'skl2onnx', 'onnxmltools'],
        'color': '#FFA07A', 'y': 4
    },
    'ONNX Hub': {
        'nodes': ['ONNX Format'],
        'color': '#FFD700', 'y': 3
    },
    'Tools': {
        'nodes': ['Netron', 'onnx-simplifier', 'onnxoptimizer', 'checker', 'shape_inf'],
        'color': '#98FB98', 'y': 3
    },
    'Runtimes': {
        'nodes': ['ORT', 'TensorRT', 'OpenVINO', 'CoreML', 'TVM'],
        'color': '#87CEEB', 'y': 2
    },
    'Hardware': {
        'nodes': ['Intel CPU', 'NVIDIA GPU', 'AMD GPU', 'Apple ANE', 'ARM CPU'],
        'color': '#DDA0DD', 'y': 1
    },
}

pos = {}
node_colors = {}

for layer_name, info in layers.items():
    nodes = info['nodes']
    y = info['y']
    color = info['color']
    n = len(nodes)
    if layer_name == 'ONNX Hub':
        pos['ONNX Format'] = (6, y)
        node_colors['ONNX Format'] = color
    elif layer_name == 'Tools':
        for i, node in enumerate(nodes):
            pos[node] = (10 + i * 1.2, y)
            node_colors[node] = color
    else:
        for i, node in enumerate(nodes):
            x = (12 - n) / 2 + i * (12 / max(n, 1))
            pos[node] = (x + 0.5, y)
            node_colors[node] = color

for node in pos:
    G.add_node(node)

fw_conv = [
    ('PyTorch', 'torch.onnx'), ('TensorFlow', 'tf2onnx'), ('JAX', 'jax2onnx'),
    ('sklearn', 'skl2onnx'), ('XGBoost', 'onnxmltools'), ('LightGBM', 'onnxmltools'),
]
for f, c in fw_conv:
    G.add_edge(f, c)

for conv in ['torch.onnx', 'tf2onnx', 'jax2onnx', 'skl2onnx', 'onnxmltools']:
    G.add_edge(conv, 'ONNX Format')

for tool in ['Netron', 'onnx-simplifier', 'onnxoptimizer', 'checker', 'shape_inf']:
    G.add_edge('ONNX Format', tool)

for rt in ['ORT', 'TensorRT', 'OpenVINO', 'CoreML', 'TVM']:
    G.add_edge('ONNX Format', rt)

rt_hw = [
    ('ORT', 'Intel CPU'), ('ORT', 'NVIDIA GPU'), ('ORT', 'AMD GPU'), ('ORT', 'ARM CPU'),
    ('TensorRT', 'NVIDIA GPU'), ('OpenVINO', 'Intel CPU'),
    ('CoreML', 'Apple ANE'), ('TVM', 'ARM CPU'), ('TVM', 'NVIDIA GPU'),
]
for r, h in rt_hw:
    G.add_edge(r, h)

fig, ax = plt.subplots(figsize=(16, 10))

nc = [node_colors.get(n, '#E8E8E8') for n in G.nodes()]
sizes = [3000 if n == 'ONNX Format' else 1500 for n in G.nodes()]

nx.draw(G, pos, ax=ax, with_labels=True, node_color=nc, node_size=sizes,
        font_size=7.5, font_weight='bold', arrows=True, arrowsize=12,
        edge_color='#AAAAAA', connectionstyle='arc3,rad=0.05', alpha=0.9,
        width=1.2)

layer_labels = [
    (0.0, 5, 'PRODUCERS', '#CC0000'),
    (0.0, 4, 'CONVERTERS', '#CC6600'),
    (0.0, 3, 'FORMAT + TOOLS', '#006600'),
    (0.0, 2, 'RUNTIMES', '#0066CC'),
    (0.0, 1, 'HARDWARE', '#660066'),
]
for x, y, label, color in layer_labels:
    ax.text(x, y, label, fontsize=9, fontweight='bold', color=color,
            va='center', ha='center',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8, edgecolor=color))

legend_elements = [
    mpatches.Patch(color='#FF6B6B', label='Training Frameworks'),
    mpatches.Patch(color='#FFA07A', label='Converters'),
    mpatches.Patch(color='#FFD700', label='ONNX Format (hub)'),
    mpatches.Patch(color='#98FB98', label='Tools'),
    mpatches.Patch(color='#87CEEB', label='Runtimes'),
    mpatches.Patch(color='#DDA0DD', label='Hardware Targets'),
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=9)
ax.set_title('Complete ONNX Ecosystem Interaction Graph', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Ecosystem graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
print(f'ONNX Format degree: in={G.in_degree("ONNX Format")}, out={G.out_degree("ONNX Format")}')
print(f'Total degree of ONNX Format: {G.in_degree("ONNX Format") + G.out_degree("ONNX Format")}')

<a id='section-10'></a>
## Section 10: Quantitative Analysis of Ecosystem Maturity

### Metcalfe's Law Applied to ONNX

The value of a network grows with the square of its participants. For the ONNX ecosystem with $N$ producers and $M$ consumers:

$$V_{\text{direct}} \propto N \cdot M$$

Without the ONNX hub, achieving this connectivity would require $N \cdot M$ custom converters. With the hub, we achieve the same connectivity with only $N + M$ converters. The **efficiency ratio** is:

$$\eta = \frac{N + M}{N \cdot M} = \frac{1}{M} + \frac{1}{N}$$

As the ecosystem grows, $\eta \to 0$, meaning the hub becomes increasingly efficient relative to point-to-point conversion.

### Ecosystem Health Metrics

We can quantify ecosystem health using several metrics:

**1. Coverage Index:** The fraction of framework-runtime pairs connected through ONNX:
$$\text{CI} = \frac{|\{(f, r) \mid f \xrightarrow{\text{ONNX}} r\}|}{|\mathcal{P}| \times |\mathcal{R}|}$$

**2. Operator Completeness:** The fraction of operators supported by each runtime:
$$\text{OC}(r) = \frac{|\text{ops}(r) \cap \mathcal{O}_{\text{onnx}}|}{|\mathcal{O}_{\text{onnx}}|}$$

**3. Ecosystem Velocity:** The rate of new component additions per year:
$$v(t) = \frac{d|\mathcal{E}|}{dt} \approx \frac{|\mathcal{E}(t)| - |\mathcal{E}(t-1)|}{1 \text{ year}}$$

**4. Hub Betweenness:** The fraction of all shortest paths passing through ONNX:
$$\beta(\text{ONNX}) = \sum_{s \neq t \neq \text{ONNX}} \frac{\sigma_{st}(\text{ONNX})}{\sigma_{st}}$$

where $\sigma_{st}$ is the number of shortest paths from $s$ to $t$ and $\sigma_{st}(\text{ONNX})$ is the number passing through ONNX.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

N_range = np.arange(2, 20)
M_range = np.arange(2, 25)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

ax1 = axes[0]
for M in [3, 5, 8, 12, 20]:
    direct = N_range * M
    hub = N_range + M
    ratio = direct / hub
    ax1.plot(N_range, ratio, 'o-', label=f'M={M}', markersize=4)

ax1.set_xlabel('Number of Producers (N)', fontsize=11)
ax1.set_ylabel('Efficiency Ratio (NM / (N+M))', fontsize=11)
ax1.set_title('Hub Efficiency vs Ecosystem Size', fontsize=12, fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)
ax1.axhline(y=1, color='red', linestyle='--', alpha=0.5, label='Break-even')

years = np.arange(2017, 2025)
components = np.array([88, 158, 221, 284, 327, 354, 394, 413])
growth_rate = np.diff(components) / components[:-1] * 100

ax2 = axes[1]
ax2.bar(years, components, color='#4ECDC4', edgecolor='black', linewidth=0.8, alpha=0.85)
ax2.set_xlabel('Year', fontsize=11)
ax2.set_ylabel('Total Ecosystem Components', fontsize=11)
ax2.set_title('Ecosystem Growth (Cumulative)', fontsize=12, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)
for x, y in zip(years, components):
    ax2.text(x, y + 5, str(y), ha='center', fontsize=8, fontweight='bold')

ax3 = axes[2]
ax3.plot(years[1:], growth_rate, 's-', color='#FF6B6B', linewidth=2, markersize=8)
ax3.fill_between(years[1:], growth_rate, alpha=0.15, color='#FF6B6B')
ax3.set_xlabel('Year', fontsize=11)
ax3.set_ylabel('YoY Growth Rate (%)', fontsize=11)
ax3.set_title('Ecosystem Velocity', fontsize=12, fontweight='bold')
ax3.grid(alpha=0.3)
for x, y in zip(years[1:], growth_rate):
    ax3.annotate(f'{y:.0f}%', (x, y), textcoords='offset points',
                xytext=(0, 10), ha='center', fontsize=8)

plt.suptitle('Quantitative Analysis of the ONNX Ecosystem', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

<a id='section-11'></a>
## Section 11: Key Properties and Guarantees

### Property 11.1: Semantic Preservation

The end-to-end pipeline must preserve model semantics. For any framework model $m_f$ and converter $\kappa_f$:

$$\forall \mathbf{x} \in \text{dom}(m_f): \|m_f(\mathbf{x}) - \rho(\gamma(\kappa_f(m_f, \mathbf{x}_s)), \mathbf{x})\|_\infty \leq \epsilon_{\text{fp}}$$

where $\epsilon_{\text{fp}}$ accounts for floating-point non-associativity across different hardware implementations. In practice, $\epsilon_{\text{fp}} \approx 10^{-5}$ for FP32 models and $\epsilon_{\text{fp}} \approx 10^{-2}$ for FP16.

### Property 11.2: Modularity (Open/Closed Principle)

The ecosystem is **open for extension** but **closed for modification**:

- Adding a new framework requires only a new converter: $\mathcal{C}' = \mathcal{C} \cup \{\kappa_{f_{N+1}}\}$
- Adding a new runtime requires only an ONNX parser: $\mathcal{R}' = \mathcal{R} \cup \{r_{M+1}\}$
- Adding a new EP requires only implementing the EP interface: $\text{EP}_{\text{new}} \subseteq \mathcal{O}_{\text{onnx}}$

No existing component needs modification. This is the architectural guarantee of the hub model.

### Property 11.3: Backward Compatibility

ONNX maintains strong backward compatibility through versioned opsets:

$$\text{valid}(\mathcal{M}, \text{opset}_v) \implies \text{valid}(\mathcal{M}, \text{opset}_{v'}) \quad \forall v' \geq v$$

Operators are never removed, only deprecated. New operator versions extend (never restrict) the input domain.

### Property 11.4: Composability

ONNX tools compose cleanly because they share the same interface type (`ModelProto → ModelProto`):

$$(\text{simplify} \circ \text{optimize} \circ \text{validate})(m) = \text{simplify}(\text{optimize}(\text{validate}(m)))$$

Any tool pipeline can be constructed by composing tools in sequence, with each tool consuming and producing the same ONNX model format. This composability is a direct consequence of the single canonical representation.

### Property 11.5: Testability

The ONNX Model Zoo serves as a **conformance test suite**. A runtime $r$ is considered conformant if:

$$\forall m \in \mathcal{Z}, \forall (\mathbf{x}_i, \mathbf{y}_i) \in \text{test\_data}(m): \|r(m, \mathbf{x}_i) - \mathbf{y}_i\|_\infty \leq \epsilon$$

This enables hardware vendors to validate their implementations against a standardized benchmark.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

frameworks = ['PyTorch', 'TensorFlow', 'JAX', 'sklearn', 'XGBoost', 'LightGBM',
              'Keras', 'PaddlePaddle', 'MATLAB', 'MXNet']
runtimes = ['ORT', 'TensorRT', 'OpenVINO', 'CoreML', 'TVM', 'NNAPI',
            'DirectML', 'WebNN']

coverage = np.array([
    [1, 1, 1, 1, 1, 1, 1, 1],   # PyTorch
    [1, 1, 1, 1, 1, 1, 1, 1],   # TensorFlow
    [1, 1, 1, 0, 1, 0, 1, 1],   # JAX
    [1, 0, 1, 0, 0, 0, 1, 0],   # sklearn
    [1, 0, 1, 0, 0, 0, 1, 0],   # XGBoost
    [1, 0, 1, 0, 0, 0, 1, 0],   # LightGBM
    [1, 1, 1, 1, 1, 1, 1, 1],   # Keras
    [1, 0, 1, 0, 1, 0, 0, 0],   # PaddlePaddle
    [1, 0, 0, 0, 0, 0, 0, 0],   # MATLAB
    [1, 0, 1, 0, 0, 0, 0, 0],   # MXNet
])

N, M = coverage.shape
ci = coverage.sum() / (N * M)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8), gridspec_kw={'width_ratios': [2, 1]})

im = ax1.imshow(coverage, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
ax1.set_xticks(range(M))
ax1.set_yticks(range(N))
ax1.set_xticklabels(runtimes, rotation=45, ha='right', fontsize=9)
ax1.set_yticklabels(frameworks, fontsize=9)

for i in range(N):
    for j in range(M):
        sym = '\u2713' if coverage[i, j] else '\u2717'
        color = 'darkgreen' if coverage[i, j] else 'darkred'
        ax1.text(j, i, sym, ha='center', va='center', fontsize=14,
                color=color, fontweight='bold')

ax1.set_title(f'ONNX Coverage Matrix (CI = {ci:.2f})', fontsize=13, fontweight='bold')
ax1.set_xlabel('Runtimes', fontsize=11)
ax1.set_ylabel('Frameworks', fontsize=11)

fw_coverage = coverage.sum(axis=1) / M
rt_coverage = coverage.sum(axis=0) / N

y_pos = np.arange(N)
bars = ax2.barh(y_pos, fw_coverage, color='#4ECDC4', edgecolor='black', linewidth=0.8)
ax2.set_yticks(y_pos)
ax2.set_yticklabels(frameworks, fontsize=9)
ax2.set_xlabel('Coverage Ratio', fontsize=10)
ax2.set_title('Per-Framework Coverage', fontsize=12, fontweight='bold')
ax2.set_xlim(0, 1.15)
ax2.grid(axis='x', alpha=0.3)

for bar, val in zip(bars, fw_coverage):
    ax2.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
             f'{val:.0%}', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

print(f'\nEcosystem Coverage Metrics:')
print(f'  Coverage Index (CI): {ci:.2%}')
print(f'  Connected pairs: {coverage.sum()} / {N * M}')
print(f'  Best-covered framework: {frameworks[np.argmax(fw_coverage)]} ({fw_coverage.max():.0%})')
print(f'  Best-covered runtime: {runtimes[np.argmax(rt_coverage)]} ({rt_coverage.max():.0%})')
print(f'  Without ONNX hub: {N * M} direct converters needed')
print(f'  With ONNX hub: {N + M} converters needed')
print(f'  Hub efficiency: {N*M / (N+M):.1f}x reduction')

<a id='section-12'></a>
## Section 12: Summary and Connections

### Core Takeaways

1. **The ONNX ecosystem is a structured collection** $\mathcal{E} = (\mathcal{P}, \mathcal{C}, \mathcal{R}, \mathcal{T}, \mathcal{Z}, \mathcal{H})$ of producers, converters, runtimes, tools, model zoo, and hardware partners — totaling 230+ actively maintained components.

2. **The deployment pipeline** $\Pi = \rho \circ \tau \circ \gamma \circ \kappa$ transforms framework models through conversion, validation, optimization, and execution stages, with semantic preservation guarantees at each transition.

3. **Converters** bridge the semantic gap between framework-specific representations and the canonical ONNX format. `torch.onnx` (tracing/dynamo), `tf2onnx` (graph conversion), and `skl2onnx` (operator mapping) represent three different conversion strategies.

4. **Runtimes** execute ONNX models with hardware-specific optimizations. ORT's **Execution Provider** architecture enables a single runtime to target diverse hardware through a plugin system with priority-based graph partitioning.

5. **Tools** form four functional categories — visualization ($\mathcal{T}_{\text{viz}}$), validation ($\mathcal{T}_{\text{val}}$), optimization ($\mathcal{T}_{\text{opt}}$), and debugging ($\mathcal{T}_{\text{dbg}}$) — that compose cleanly through the shared `ModelProto` interface.

6. **The Model Zoo** provides ~180 pre-trained models across 8 categories, serving as baselines, deployment templates, and conformance test suites.

7. **Hub efficiency** grows with ecosystem size: the $N + M$ hub model is $\frac{NM}{N+M}\times$ more efficient than $N \times M$ point-to-point conversion.

### Mathematical Framework Summary

$$\boxed{\text{ONNX Ecosystem} = \underbrace{\mathcal{P} \xrightarrow{\mathcal{C}} \text{ONNX}}_{\text{production}} \xrightarrow{\mathcal{T}} \underbrace{\text{ONNX}_{\text{opt}} \xrightarrow{\mathcal{R}} \mathcal{H}}_{\text{consumption}}}$$

### Connections to Subsequent Topics

| This Section | Next Sections |
|:---|:---|
| Converter pipeline overview | → Module 3: ONNX Architecture (full protobuf anatomy) |
| Runtime feature matrix | → Module 8: Model Optimization (graph opts, quantization) |
| EP architecture | → Module 5: ONNX Runtime Deep Dive (EP internals) |
| Model Zoo overview | → Apply notebook: hands-on with zoo models |
| Tools ecosystem | → Apply notebook: using simplifier, inspector, benchmarks |

---

**Next:** [ONNX Ecosystem Overview — Apply (Hands-On)](./ONNX_Ecosystem_Overview_Apply.ipynb) | [Installation and Setup](../04_Installation_and_Setup/)